In [ ]:
import serial
import serial.tools.list_ports
import threading
import time
from collections import deque
import numpy as np

import matplotlib
matplotlib.use("TkAgg")  # GUI-backend
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────────────────────────
# Lyd (valgfrit)
TRY_AUDIO = True
AUDIO_AVAILABLE = False
VOLUME = 0.5      # 0.0 … 1.0
MUTED  = False

try:
    if TRY_AUDIO:
        import sounddevice as sd
        AUDIO_AVAILABLE = True
except Exception as _e:
    AUDIO_AVAILABLE = False
    print("sounddevice ikke tilgængelig – kører uden audio:", repr(_e))

# ──────────────────────────────────────────────────────────────────────────────

# Debug-udskrivning af samples
DEBUG_PRINT_SAMPLES   = True   # slå fra/til
PRINT_EVERY_N_BLOCKS  = 25     # print hver N'te modtagne blok
SAMPLES_TO_PRINT      = 16     # hvor mange samples der printes fra blokken

# KONFIGURATION — tilpas efter behov
VIDPID               = "16C0:0483"    # Teensy 4.1
BAUDRATE             = 9600           # hvis USB Serial: værdi er ligegyldig, men behold 9600
TIMEOUT              = 2.0            # seriel timeout (sek)
START_BLOCKS         = 65535          # hvor mange blokke Teensy skal sende (kan gentages automatisk)
USE_NEWLINE          = True           # True hvis Teensy læser linjer (readStringUntil('\n'))

SAMPLE_RATE_HZ       = 15000          # kun til akse-skala / frekvensakse og lydafspilning
MAX_SECONDS_BUFFER   = 3.0            # hvor meget tid vi gemmer i ringbuffer
EXPECTED_ADC_BITS    = 12             # skala omkring midtpunkt for DC-fjernelse

# FFT / visning
FFT_N                = 4096
REFRESH_HZ           = 10             # antal GUI-opdateringer pr sekund
WINDOW               = "hann"         # "hann", "hamming", "blackman", None
AVG_EMA              = 8              # glidende gennemsnit af FFT (1 = fra)

# Debug
PRINT_FIRST_BYTES    = False          # sæt True for at sniffe de første bytes efter connect

# ──────────────────────────────────────────────────────────────────────────────
stop_event = threading.Event()
ser_lock   = threading.Lock()
ser        = None

RING_CAP   = max(FFT_N, int(MAX_SECONDS_BUFFER * SAMPLE_RATE_HZ))
ringbuf    = deque(maxlen=RING_CAP)

# Audio-kø (separat fra ringbuffer til visning)
AUDIO_Q_MAX_SEC = 2.0
audio_q = deque(maxlen=int(AUDIO_Q_MAX_SEC * SAMPLE_RATE_HZ))  # holder rå int16 samples

# ── Digitalt lavpas (viewer) ──────────────────────────────────
USE_VIEW_LPF  = True        # ← Sæt til False for at slå filter fra
DOWNSAMPLE    = 8           # bruges kun til at sætte cutoff korrekt
DECIM_TAPS    = 511 #255         # 127/255/321 ... flere tap = skarpere, mere CPU
DECIM_WINDOW  = "blackman"  # "blackman", "hann", "hamming", None
PASS_FRAC     = 0.9         # cutoff = PASS_FRAC * (1/D) * Nyquist

# FIR-design: windowed-sinc lavpas
def design_lowpass_taps(num_taps, D, window="blackman", pass_frac=0.9):
    # fc_nyq er fraktion af Nyquist; konverter til fraktion af Fs (0..0.5)
    fc_nyq = pass_frac / float(D)
    fc_fs  = fc_nyq / 2.0                      # <<< vigtigt: 2×-fix

    n = np.arange(num_taps) - (num_taps - 1) / 2.0
    h = 2 * fc_fs * np.sinc(2 * fc_fs * n)     # ideal LPF (normaliseret til Fs)
    wname = (window or "").lower()
    if   wname == "blackman": w = np.blackman(num_taps)
    elif wname == "hann":     w = np.hanning(num_taps)
    elif wname == "hamming":  w = np.hamming(num_taps)
    else:                     w = np.ones(num_taps)
    h *= w
    h /= h.sum() + 1e-20                       # DC-gain = 1
    return h.astype(np.float64)

def _report_fc(h, fs):
    N = 1 << 15
    H = np.fft.rfft(h, N)
    f = np.fft.rfftfreq(N, 1.0/fs)
    mag = 20*np.log10(np.abs(H) / (np.abs(H[0]) + 1e-20) + 1e-20)
    idx = np.argmax(mag <= -3.0)
    print(f"-3 dB ≈ {f[idx]:.1f} Hz")

# FIR-state (initialiseres i main)
lpf_h = None
lpf_state = None


# FFT vindue
def get_window(n, kind):
    if not kind or str(kind).lower() == "none":
        return np.ones(n, dtype=np.float64)
    k = str(kind).lower()
    if k == "hann":
        return np.hanning(n)
    if k == "hamming":
        return np.hamming(n)
    if k == "blackman":
        return np.blackman(n)
    return np.ones(n, dtype=np.float64)

WIN = get_window(FFT_N, WINDOW)
WIN_ENERGY = np.sqrt((WIN**2).mean())

avg_accum = None  # til EMA af FFT
ADC_MAX   = (1 << EXPECTED_ADC_BITS) - 1

SELECT_ADC = 0   # 0 = den “rigtige” kanal med stor spredning. Skift til 1 hvis du vil se adc=1.
SPEC_SCALE = "dBFS"   # vælg: "dBFS", "rms", "psd"
FULL_SCALE_COUNTS = ADC_MAX / 2.0  # peak af centreret 12-bit ADC ≈ 2047

# ──────────────────────────────────────────────────────────────────────────────
# Seriel-hjælpere
def find_port(vidpid=VIDPID):
    want = vidpid.replace("VID:PID=","").upper()
    for p in serial.tools.list_ports.comports():
        hw = (p.hwid or "").upper()
        if want in hw or "VID:PID="+want in hw or "16C0:0483" in hw:
            return p.device
    raise IOError("Teensy ikke fundet på serielle porte")

def open_serial():
    while not stop_event.is_set():
        try:
            port = find_port()
            s = serial.Serial(port, baudrate=BAUDRATE, timeout=TIMEOUT)
            # host-ready signaler — ofte nødvendigt for Teensy-sketches
            s.dtr = True
            s.rts = True
            time.sleep(0.1)
            s.reset_input_buffer()
            if PRINT_FIRST_BYTES:
                s.timeout = 0.5
                sniff = s.read(16)
                print("Sniffer første 16 bytes:", sniff)
                s.timeout = TIMEOUT
            return s
        except Exception as e:
            print("Venter på Teensy…", e)
            time.sleep(1.0)
    raise InterruptedError("Afbrudt inden seriel åbnede")

def send_start(s):
    cmd = f"START {START_BLOCKS}"
    if USE_NEWLINE:
        cmd += "\n"
    with ser_lock:
        s.write(cmd.encode("ascii", errors="ignore"))
        s.flush()

def send_stop(s):
    with ser_lock:
        try:
            s.write(b"STOP\n" if USE_NEWLINE else b"STOP")
            s.reset_input_buffer()
        except Exception:
            pass

def sync(s, timeout_s=3.0):
    t0 = time.time()
    while not stop_event.is_set():
        b = s.read(1)
        if b == b'\x7E':
            return True
        if (time.time() - t0) > timeout_s and timeout_s > 0:
            return False
    return False

def read_block(s):
    """Læser 0x7E + 7 byte header + count*2 data; returnerer (adc_id, block_id, np.uint16 array)."""
    if not sync(s):
        raise RuntimeError("Sync timeout — fik ikke 0x7E")

    hdr = s.read(7)
    if len(hdr) != 7:
        raise RuntimeError("Header timeout")

    adc, hi, lo, b3, b2, b1, b0 = hdr
    count = (hi << 8) | lo
    block_id = (b3 << 24) | (b2 << 16) | (b1 << 8) | b0

    need = count * 2
    buf = bytearray()
    while len(buf) < need and not stop_event.is_set():
        chunk = s.read(need - len(buf))
        if not chunk:
            continue
        buf.extend(chunk)

    if len(buf) < need:
        raise RuntimeError("Data timeout")

    data = np.frombuffer(bytes(buf), dtype=np.uint16)
    return adc, block_id, data

# ──────────────────────────────────────────────────────────────────────────────
# Reader-tråd
def reader_thread():
    global ser
    try:
        send_start(ser)
    except Exception:
        pass

    while not stop_event.is_set():
        try:
            adc, block_id, u16 = read_block(ser)
            if adc != SELECT_ADC:
                continue
        except RuntimeError:
            # prøv at kickstarte stream igen (fx når START_BLOCKS løber tør)
            try:
                send_start(ser)
            except Exception:
                pass
            continue
        except (serial.SerialException, OSError):
            break

        # ── ALT HERUNDER SKAL VÆRE INDE I while-LOOP’ET ──
        global lpf_h, lpf_state

        # Konverter til centreret int16 med DC fjernet
        x_i16 = (u16.astype(np.int32) - (ADC_MAX // 2))
        x_i16 = np.clip(x_i16, -32768, 32767).astype(np.int16)

        if USE_VIEW_LPF:
            # Streamende FIR: brug tidligere state + ny blok
            xin = x_i16.astype(np.float64)
            z   = np.concatenate((lpf_state, xin))                 # len = (taps-1) + N
            y   = np.convolve(z, lpf_h, mode='valid')              # len = N
            lpf_state = z[-(DECIM_TAPS - 1):].copy()               # opdatér state
            x_out = np.clip(np.rint(y), -32768, 32767).astype(np.int16)
        else:
            x_out = x_i16

        # Til ringbuffer (visning)
        ringbuf.extend(x_out)

        # Til lydkø (afspilning)
        if AUDIO_AVAILABLE and not MUTED:
            free = audio_q.maxlen - len(audio_q)
            if free > 0:
                take = min(len(x_out), free)
                audio_q.extend(x_out[:take])


    # pæn oprydning
    try:
        send_stop(ser)
    except Exception:
        pass

# ──────────────────────────────────────────────────────────────────────────────
# FFT
def compute_fft(x_i16):
    x = x_i16.astype(np.float64)
    x -= x.mean()

    w  = WIN
    N  = FFT_N
    Fs = SAMPLE_RATE_HZ

    xw = x * w
    X  = np.fft.rfft(xw, n=N)

    # Vindueskonstanter
    CG   = w.sum() / N                 # coherent gain (Hann ≈ 0.5)
    U    = (w**2).sum() / N            # mean-square of window

    freqs = np.fft.rfftfreq(N, 1.0/Fs)

    if SPEC_SCALE.lower() == "psd":
        Sxx = (np.abs(X)**2) / (Fs * (N**2) * U)   # to-sidet
        Sxx[1:-1] *= 2.0                           # én-sidet
        ASD = np.sqrt(Sxx)                         # counts RMS / √Hz
        ref = (FULL_SCALE_COUNTS/np.sqrt(2))       # FS-sinus RMS
        mag_db = 20*np.log10(ASD/ ref + 1e-20)
        ylabel = "ASD [dBFS/√Hz]"
        return freqs, mag_db, ylabel

    # Én-sidet amplitudespektrum (peak)
    A = np.abs(X) / (N * CG)        # to-sidet peak
    A[1:-1] *= 2.0                   # én-sidet peak

    if SPEC_SCALE.lower() == "rms":
        mag_db = 20*np.log10(A/np.sqrt(2) + 1e-20)   # counts RMS
        ylabel = "Amplitude [dB counts RMS]"
        return freqs, mag_db, ylabel

    # dBFS (RMS) – 0 dBFS ~ FS-sinus RMS
    A_rms = A/np.sqrt(2)
    ref   = (FULL_SCALE_COUNTS/np.sqrt(2))
    mag_db = 20*np.log10(A_rms/ref + 1e-20)
    ylabel = "Amplitude [dBFS]"
    return freqs, mag_db, ylabel

# ──────────────────────────────────────────────────────────────────────────────
# Lyd: callback-baseret streaming
def make_audio_stream():
    """Returnerer en sounddevice.OutputStream som spiller fra audio_q."""
    if not AUDIO_AVAILABLE:
        return None

    def callback(outdata, frames, time_info, status):
        global VOLUME, MUTED
        if status:
            # Undgå spam, men vis underruns/overruns
            print("Audio status:", status)

        # Hent 'frames' samples fra køen
        if MUTED or len(audio_q) == 0:
            # bare stilhed
            outdata[:] = 0
            return

        # Træk samples — deque af int16; vi skaber en lokal buffer
        n = min(frames, len(audio_q))
        # Hent n samples
        # Effektivt: brug numpy til at konvertere listeudlæsning
        buf = np.empty(frames, dtype=np.int16)
        # Fyld det vi har
        for i in range(n):
            buf[i] = audio_q.popleft()
        # Hvis mangler, pad med nul
        if n < frames:
            buf[n:] = 0

        # Volume (multiplicer i int32 for at undgå wrap)
        if VOLUME != 1.0:
            tmp = (buf.astype(np.int32) * float(VOLUME))
            np.clip(tmp, -32768, 32767, out=tmp)
            buf = tmp.astype(np.int16)

        # Mono -> (frames, 1)
        outdata[:, 0] = buf

    # Opret stream (mono, int16)
    stream = sd.OutputStream(
        samplerate=SAMPLE_RATE_HZ,
        channels=1,
        dtype='int16',
        blocksize=0,     # lad backend vælge
        callback=callback
    )
    return stream

# ──────────────────────────────────────────────────────────────────────────────
# GUI / plots
def main():
    global ser, avg_accum, VOLUME, MUTED

    ser = open_serial()
    print(f">> Forbundet til Teensy på {ser.port}")
    
    global lpf_h, lpf_state
    if USE_VIEW_LPF:
        lpf_h = design_lowpass_taps(DECIM_TAPS, DOWNSAMPLE, DECIM_WINDOW, PASS_FRAC)
        lpf_state = np.zeros(DECIM_TAPS - 1, dtype=np.float64)
        print(f"Viewer LPF: taps={DECIM_TAPS}, window={DECIM_WINDOW}, cutoff≈{PASS_FRAC/DOWNSAMPLE:.4f}·Nyquist")
        _report_fc(lpf_h, SAMPLE_RATE_HZ)
        
    th = threading.Thread(target=reader_thread, daemon=True)
    th.start()

    # Start audio (hvis muligt)
    # Start audio (hvis muligt)
    audio_stream = None
    if AUDIO_AVAILABLE:
        try:
            audio_stream = make_audio_stream()
            if audio_stream is not None:
                audio_stream.start()
                print("Audio startet (m='mute', [ og ] = volume ned/op, f = filter til/fra)")
        except Exception as e:
            print("Kunne ikke starte audio stream:", repr(e))
            audio_stream = None
    else:
        print("Taster: m='mute', [ og ] = volume ned/op, f = filter til/fra")
    

    plt.figure("Teensy Scope", figsize=(11, 7))

    ax_time = plt.subplot(3,1,1)
    ax_time.set_title(f"Tidsdomæne (ADC={SELECT_ADC}, seneste samples)")
    ax_time.set_xlabel("Tid [s]")
    ax_time.set_ylabel("Amplitude (rel.)")
    line_time, = ax_time.plot([], [])

    ax_hist = plt.subplot(3,1,2)
    ax_hist.set_title("Histogram (fordeling af samples)")
    ax_hist.set_xlabel("Amplitude (rel.)")
    ax_hist.set_ylabel("Tællinger")
    hist_bins = 128
    bars = None  # opdateres dynamisk

    ax_fft = plt.subplot(3,1,3)
    ax_fft.set_title(f"FFT (ADC={SELECT_ADC}, N={FFT_N}, vindue={WINDOW}, EMA={AVG_EMA}×, scale={SPEC_SCALE})")
    ax_fft.set_xlabel("Frekvens [Hz]")
    ax_fft.set_ylabel("Magnitude [dB]")
    freqs_init = np.fft.rfftfreq(FFT_N, d=1.0/SAMPLE_RATE_HZ)
    line_fft, = ax_fft.plot(freqs_init, np.full_like(freqs_init, -200.0))
    ax_fft.set_xlim(0, SAMPLE_RATE_HZ/2)
    ax_fft.set_ylim(-160, 20)

    plt.tight_layout()

    refresh_period = 1.0 / max(1, REFRESH_HZ)
    last = time.time()

    def on_close(evt=None):
        stop_event.set()
        try:
            if th.is_alive():
                th.join(timeout=TIMEOUT + 1.0)
        except Exception:
            pass
        try:
            if audio_stream is not None:
                audio_stream.stop()
                audio_stream.close()
        except Exception:
            pass
        try:
            with ser_lock:
                if ser and ser.is_open:
                    ser.close()
        except Exception:
            pass
        plt.close("all")

    def on_key(event):
        global VOLUME, MUTED, USE_VIEW_LPF, lpf_h, lpf_state
        if event.key == 'm':
            MUTED = not MUTED
            print("Mute:", MUTED)
        elif event.key == ']':
            VOLUME = min(1.0, VOLUME + 0.05); print(f"Volume: {VOLUME:.2f}")
        elif event.key == '[':
            VOLUME = max(0.0, VOLUME - 0.05); print(f"Volume: {VOLUME:.2f}")
        elif event.key == 'f':
            USE_VIEW_LPF = not USE_VIEW_LPF
            if USE_VIEW_LPF:
                lpf_h = design_lowpass_taps(DECIM_TAPS, DOWNSAMPLE, DECIM_WINDOW, PASS_FRAC)
                lpf_state = np.zeros(DECIM_TAPS - 1, dtype=np.float64)
                _report_fc(lpf_h, SAMPLE_RATE_HZ)
            print("LPF:", "ON" if USE_VIEW_LPF else "OFF")
    

    def update(_=None):
        global avg_accum          # <— til EMA buffer
        nonlocal bars, last
        if stop_event.is_set():
            return

        now = time.time()
        if now - last < refresh_period:
            plt.gcf().canvas.new_timer(interval=int(1000/REFRESH_HZ)).start()
            return
        last = now

        n = len(ringbuf)
        if n < FFT_N:
            plt.gcf().canvas.new_timer(interval=int(1000/REFRESH_HZ)).start()
            return

        # tag en blok til tidsplot (~1/REFRESH_HZ af sample rate eller max RING_CAP)
        show_n = min(n, int(SAMPLE_RATE_HZ / max(1, REFRESH_HZ)))
        x = np.frombuffer(np.array(ringbuf, dtype=np.int16)[-show_n:].tobytes(), dtype=np.int16)

        # tidsakse
        t = np.arange(show_n) / float(SAMPLE_RATE_HZ)
        line_time.set_data(t, x)
        ax_time.relim(); ax_time.autoscale_view()

        # histogram
        h_vals, edges = np.histogram(x, bins=hist_bins, range=(-ADC_MAX//2, ADC_MAX//2))
        centers = (edges[:-1] + edges[1:]) / 2.0
        if bars is None:
            bars = ax_hist.bar(centers, h_vals, width=edges[1]-edges[0], align='center')
        else:
            for b, h in zip(bars, h_vals):
                b.set_height(int(h))
        ax_hist.relim(); ax_hist.autoscale_view()

        # FFT på de seneste FFT_N samples
        x_fft = np.frombuffer(np.array(ringbuf, dtype=np.int16)[-FFT_N:].tobytes(), dtype=np.int16)
        freqs, mag_db, ylab = compute_fft(x_fft)
        ax_fft.set_ylabel(ylab)

        # EMA glidende gennemsnit (lin-domæne)
        if AVG_EMA > 1:
            mag_lin = 10.0 ** (mag_db / 20.0)
            alpha = 1.0 / AVG_EMA
            if (avg_accum is None) or (avg_accum.shape != mag_lin.shape):
                avg_accum = mag_lin.copy()
            else:
                avg_accum = (1.0 - alpha) * avg_accum + alpha * mag_lin
            mag_db = 20.0 * np.log10(avg_accum + 1e-20)

        line_fft.set_data(freqs, mag_db)
        ax_fft.set_xlim(0, SAMPLE_RATE_HZ/2)
        y_min, y_max = float(np.nanmin(mag_db)), float(np.nanmax(mag_db))
        margin = 10.0
        ax_fft.set_ylim(max(-200, y_min - margin), min(20, y_max + margin))

        plt.gcf().canvas.draw_idle()
        plt.gcf().canvas.new_timer(interval=int(1000/REFRESH_HZ)).start()

    fig = plt.gcf()
    fig.canvas.mpl_connect('close_event', on_close)
    fig.canvas.mpl_connect('key_press_event', on_key)

    # periodisk opdatering
    timer = fig.canvas.new_timer(interval=int(1000/REFRESH_HZ))
    timer.add_callback(update)
    timer.start()
    plt.show()
    on_close()

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        pass
